# NFStreamer-Compatible AE + RF Fusion Training Notebook

This notebook trains and evaluates a deployment-oriented AE + RF fusion model using **30 realtime-extractable NFStreamer-compatible features**.

Output folder:

```text
models_nfstream/
├── scaler.joblib
├── ae_model.pth
├── rf_model.joblib
├── feature_30.json
├── feature_20.json
└── deployment_metadata.json
```

In [ ]:
# Cell 1: Setup and Environment Initialization

%load_ext autoreload
%autoreload 2

import sys
import json
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

current_dir = Path.cwd()
root_dir = current_dir.parent
if str(root_dir) not in sys.path:
    sys.path.append(str(root_dir))

print(f"Project Root: {root_dir}")

from src import config, preprocessing, autoencoder, rf_classifier, evaluation, utils

print("Checking Data Directories:")
print(f"  - 2017: {config.DIR_2017} -> {'Found' if config.DIR_2017.exists() else 'Not Found'}")
print(f"  - 2018: {config.DIR_2018} -> {'Found' if config.DIR_2018.exists() else 'Not Found'}")

deploy_dir = root_dir / "models_nfstream"
deploy_dir.mkdir(parents=True, exist_ok=True)

eval_dir = deploy_dir / "evaluation"
eval_dir.mkdir(parents=True, exist_ok=True)

print(f"NFStream-compatible model directory: {deploy_dir}")
print(f"Evaluation output directory: {eval_dir}")
print(f"Device from config: {config.DEVICE}")

In [ ]:
# Cell 2: Define NFStreamer-Compatible Feature Set

NFSTREAM_SELECTED_FEATURES_30 = [
    'RST Flag Count',
    'Total Length of Fwd Packets',
    'Bwd IAT Min',
    'ECE Flag Count',
    'act_data_pkt_fwd',
    'Bwd Packet Length Min',
    'Total Fwd Packets',
    'Bwd IAT Mean',
    'PSH Flag Count',
    'Destination Port',
    'Flow IAT Std',
    'Bwd Packet Length Std',
    'Bwd IAT Max',
    'Fwd PSH Flags',
    'Fwd Packet Length Max',
    'Flow Duration',
    'SYN Flag Count',
    'Fwd IAT Min',
    'Bwd IAT Std',
    'Down/Up Ratio',
    'Fwd IAT Total',
    'Fwd Packet Length Std',
    'Fwd IAT Mean',
    'URG Flag Count',
    'Min Packet Length',
    'Bwd Packets/s',
    'Packet Length Std',
    'Flow IAT Max',
    'Fwd Packet Length Mean',
    'Fwd IAT Max'
]

NFSTREAM_MRMR_FEATURES_20 = NFSTREAM_SELECTED_FEATURES_30[:20]

FEATURE_30 = NFSTREAM_SELECTED_FEATURES_30
FEATURE_20 = NFSTREAM_MRMR_FEATURES_20

assert len(FEATURE_30) == 30, "FEATURE_30 must contain exactly 30 features."
assert len(FEATURE_20) == 20, "FEATURE_20 must contain exactly 20 features."

print(f"Number of NFStreamer-compatible features: {len(FEATURE_30)}")
print(f"Number of primary fusion features: {len(FEATURE_20)}")
print("\nFEATURE_30:")
for i, f in enumerate(FEATURE_30, 1):
    print(f"{i:02d}. {f}")

In [ ]:
# Cell 3: Load 2017 and 2018 Datasets Separately

print("Loading 2017 and 2018 datasets separately...")

X_17, y_17 = preprocessing.load_single_dataset_year("2017", binary_mode=True)
print(f"Loaded 2017: X={X_17.shape}, y={y_17.shape}")

X_18, y_18 = preprocessing.load_single_dataset_year("2018", binary_mode=True)
print(f"Loaded 2018: X={X_18.shape}, y={y_18.shape}")

assert list(X_17.columns) == config.SELECTED_FEATURES, "X_17 columns do not match config.SELECTED_FEATURES"
assert list(X_18.columns) == config.SELECTED_FEATURES, "X_18 columns do not match config.SELECTED_FEATURES"

print("Feature order verified: both datasets follow config.SELECTED_FEATURES.")

In [ ]:
# Cell 4: Verify NFStreamer-Compatible Feature Availability in Dataset

missing_from_2017 = [f for f in FEATURE_30 if f not in X_17.columns]
missing_from_2018 = [f for f in FEATURE_30 if f not in X_18.columns]
missing_20 = [f for f in FEATURE_20 if f not in FEATURE_30]

if missing_from_2017:
    raise ValueError(f"Missing from X_17: {missing_from_2017}")
if missing_from_2018:
    raise ValueError(f"Missing from X_18: {missing_from_2018}")
if missing_20:
    raise ValueError(f"FEATURE_20 contains items not in FEATURE_30: {missing_20}")

print("All NFStreamer-compatible features are available in the loaded datasets.")

In [ ]:
# Cell 5: Independent Splitting and Mixed Training Construction

print("Splitting 2017 and 2018 independently with stratification...")

X_17_train, X_17_test, y_17_train, y_17_test = train_test_split(
    X_17, y_17, test_size=0.2, random_state=config.SEED, stratify=y_17
)

X_18_train, X_18_test, y_18_train, y_18_test = train_test_split(
    X_18, y_18, test_size=0.2, random_state=config.SEED, stratify=y_18
)

X_train_df = pd.concat([X_17_train, X_18_train], ignore_index=True)
y_train = pd.concat([pd.Series(y_17_train), pd.Series(y_18_train)], ignore_index=True)

X_test_all_df = pd.concat([X_17_test, X_18_test], ignore_index=True)
y_test_all = pd.concat([pd.Series(y_17_test), pd.Series(y_18_test)], ignore_index=True)

print(f"Mixed Train Size: {X_train_df.shape}")
print(f"Test 2017 Size:  {X_17_test.shape}")
print(f"Test 2018 Size:  {X_18_test.shape}")
print(f"Global Test Size: {X_test_all_df.shape}")

print("\nTraining label distribution:")
print(y_train.value_counts())

del X_17, X_18, X_17_train, X_18_train
gc.collect()

In [ ]:
# Cell 6: Select 30 NFStreamer-Compatible Features

print("Selecting 30 NFStreamer-compatible features...")

X_train_30 = X_train_df[FEATURE_30].copy()
X_17_test_30 = X_17_test[FEATURE_30].copy()
X_18_test_30 = X_18_test[FEATURE_30].copy()
X_test_all_30 = X_test_all_df[FEATURE_30].copy()

X_train_30 = X_train_30.replace([np.inf, -np.inf], np.nan).fillna(0)
X_17_test_30 = X_17_test_30.replace([np.inf, -np.inf], np.nan).fillna(0)
X_18_test_30 = X_18_test_30.replace([np.inf, -np.inf], np.nan).fillna(0)
X_test_all_30 = X_test_all_30.replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"X_train_30: {X_train_30.shape}")
print(f"X_17_test_30: {X_17_test_30.shape}")
print(f"X_18_test_30: {X_18_test_30.shape}")
print(f"X_test_all_30: {X_test_all_30.shape}")

assert X_train_30.shape[1] == 30
assert list(X_train_30.columns) == FEATURE_30

In [ ]:
# Cell 7: Fit and Save 30-Feature Scaler

print("Scaling 30 NFStreamer-compatible features...")

scaler_30 = StandardScaler()

X_train_30_scaled = scaler_30.fit_transform(X_train_30)
X_17_test_30_scaled = scaler_30.transform(X_17_test_30)
X_18_test_30_scaled = scaler_30.transform(X_18_test_30)
X_test_all_30_scaled = scaler_30.transform(X_test_all_30)

scaler_path = deploy_dir / "scaler.joblib"
joblib.dump(scaler_30, scaler_path)

print(f"Saved scaler to: {scaler_path}")
print("Scaler n_features_in_:", getattr(scaler_30, "n_features_in_", None))

In [ ]:
# Cell 8: Train Autoencoder on 30 Scaled NFStreamer-Compatible Features

print(f"Training Autoencoder on device: {config.DEVICE}")

ae_model = autoencoder.DeepAutoencoder(input_dim=30, latent_dim=5, hidden_layers=[22, 12])
ae_save_path = deploy_dir / "ae_model.pth"

ae_model = autoencoder.train_ae(
    ae_model,
    X_train_30_scaled,
    save_path=ae_save_path
)

print("AE training completed.")
print(f"Saved AE model to: {ae_save_path}")

In [ ]:
# Cell 9: Extract Latent Features from Autoencoder Encoder

print("Extracting 5-dimensional latent features...")

X_train_latent = autoencoder.extract_features(ae_model, X_train_30_scaled)
X_17_test_latent = autoencoder.extract_features(ae_model, X_17_test_30_scaled)
X_18_test_latent = autoencoder.extract_features(ae_model, X_18_test_30_scaled)
X_test_all_latent = autoencoder.extract_features(ae_model, X_test_all_30_scaled)

print(f"X_train_latent: {X_train_latent.shape}")
print(f"X_17_test_latent: {X_17_test_latent.shape}")
print(f"X_18_test_latent: {X_18_test_latent.shape}")
print(f"X_test_all_latent: {X_test_all_latent.shape}")

assert X_train_latent.shape[1] == 5

In [ ]:
# Cell 10: Fusion Strategy — 20 Primary Features + 5 Latent Features

print("Executing fusion strategy: 20 primary features + 5 latent features...")

idx_20 = [FEATURE_30.index(f) for f in FEATURE_20]

X_train_20_scaled = X_train_30_scaled[:, idx_20]
X_17_test_20_scaled = X_17_test_30_scaled[:, idx_20]
X_18_test_20_scaled = X_18_test_30_scaled[:, idx_20]
X_test_all_20_scaled = X_test_all_30_scaled[:, idx_20]

X_train_fusion = np.hstack([X_train_20_scaled, X_train_latent])
X_17_test_fusion = np.hstack([X_17_test_20_scaled, X_17_test_latent])
X_18_test_fusion = np.hstack([X_18_test_20_scaled, X_18_test_latent])
X_test_all_fusion = np.hstack([X_test_all_20_scaled, X_test_all_latent])

print(f"X_train_fusion: {X_train_fusion.shape}")
print(f"X_17_test_fusion: {X_17_test_fusion.shape}")
print(f"X_18_test_fusion: {X_18_test_fusion.shape}")
print(f"X_test_all_fusion: {X_test_all_fusion.shape}")

assert X_train_fusion.shape[1] == 25

In [ ]:
# Cell 11: Train Random Forest on 25-Dimensional Fusion Vector

print("Training Random Forest on fusion data...")

rf_save_path = deploy_dir / "rf_model.joblib"

rf_model = rf_classifier.train_rf(
    X_train_fusion,
    y_train,
    save_path=rf_save_path
)

print("RF training completed.")
print(f"Saved RF model to: {rf_save_path}")
print("RF n_features_in_:", getattr(rf_model, "n_features_in_", None))

In [ ]:
# Cell 12: Evaluate NFStreamer-Compatible Deployment Model

print("\n--- STARTING NFSTREAM-COMPATIBLE MODEL EVALUATION ---")

test_scenarios = [
    (X_17_test_fusion, y_17_test, "NFStream30_Unseen_2017"),
    (X_18_test_fusion, y_18_test, "NFStream30_Unseen_2018"),
    (X_test_all_fusion, y_test_all, "NFStream30_Global_Mixed_Test"),
]

for X_t, y_t, name in test_scenarios:
    print(f"\nEvaluating Scenario: {name}")
    evaluation.evaluate_model(
        model=rf_model,
        X_test=X_t,
        y_test=y_t,
        save_dir=eval_dir,
        dataset_name=name
    )

In [ ]:
# Cell 13: Save Feature Lists and Deployment Metadata

feature_30_path = deploy_dir / "feature_30.json"
feature_20_path = deploy_dir / "feature_20.json"
metadata_path = deploy_dir / "deployment_metadata.json"

with open(feature_30_path, "w", encoding="utf-8") as f:
    json.dump(FEATURE_30, f, indent=2, ensure_ascii=False)

with open(feature_20_path, "w", encoding="utf-8") as f:
    json.dump(FEATURE_20, f, indent=2, ensure_ascii=False)

metadata = {
    "description": "NFStreamer-compatible AE-RF fusion model using 30 realtime-extractable features.",
    "scaler_input_dim": 30,
    "ae_input_dim": 30,
    "ae_latent_dim": 5,
    "rf_input_dim": 25,
    "fusion": "20 primary scaled NFStreamer-compatible features + 5 Autoencoder latent features",
    "feature_30": FEATURE_30,
    "feature_20": FEATURE_20,
    "seed": config.SEED,
    "rf_estimators": config.RF_ESTIMATORS,
    "rf_max_depth": config.RF_MAX_DEPTH,
    "ae_hidden_layers": [22, 12],
    "ae_epochs": config.AE_EPOCHS,
    "ae_batch_size": config.AE_BATCH_SIZE,
    "ae_lr": config.AE_LR,
}

with open(metadata_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)

print("Saved:")
print(" -", feature_30_path)
print(" -", feature_20_path)
print(" -", metadata_path)

In [ ]:
# Cell 14: Sanity Check Saved Artifacts

print("Running sanity check on saved NFStreamer-compatible artifacts...")

loaded_scaler = joblib.load(deploy_dir / "scaler.joblib")
loaded_rf = joblib.load(deploy_dir / "rf_model.joblib")

print("Scaler n_features_in_:", getattr(loaded_scaler, "n_features_in_", None))
print("RF n_features_in_:", getattr(loaded_rf, "n_features_in_", None))

assert getattr(loaded_scaler, "n_features_in_", None) == 30, "Scaler must expect 30 features."
assert getattr(loaded_rf, "n_features_in_", None) == 25, "RF must expect 25 fused features."

state = torch.load(deploy_dir / "ae_model.pth", map_location="cpu")
print("\nAE state_dict shapes:")
for k, v in state.items():
    if hasattr(v, "shape"):
        print(k, tuple(v.shape))

print("\nNFStreamer-compatible deployment artifacts are consistent.")
print(f"Copy this folder to the VM if needed: {deploy_dir}")

In [ ]:
# Cell 15: Realtime Predictor Command

print("After copying models_nfstream/ to the VM, run this in the realtime folder:")
print()
print("python3 predictor.py \\")
print("  --scaler ../models_nfstream/scaler.joblib \\")
print("  --ae ../models_nfstream/ae_model.pth \\")
print("  --rf ../models_nfstream/rf_model.joblib \\")
print("  --dry-run")